In [ ]:
%pip install datasets scikit-learn

In [4]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

In [6]:
dataset = load_dataset('sms_spam')
dataset

DatasetDict({
    train: Dataset({
        features: ['sms', 'label'],
        num_rows: 5574
    })
})

In [13]:
dataset = dataset['train'].train_test_split(test_size=0.2) # Datensatz split
dataset["train"]

Dataset({
    features: ['sms', 'label'],
    num_rows: 1168
})

In [14]:
# Vektorisierung 
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(max_features=1000) #  [100, 200, ....,] 1000 häufigste Wörter
X_train = vectorizer.fit_transform([x['sms'] for x in dataset['train']]).toarray()
X_test = vectorizer.transform([x['sms'] for x in dataset['test']]).toarray()
y_train = [x['label'] for x in dataset['train']]
y_test = [x['label'] for x in dataset['test']]


array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,

In [15]:
import torch
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)


In [20]:
import torch.nn as nn
import torch.optim as optim


class SpamClassifier(nn.Module):

    def __init__(self,input_dim): # Konstruktor
        super().__init__() # Superklasse init
        # Fully connected feed-forward nn
        self.fc1 = nn.Linear(input_dim, 32) # immer Zweierpotenzen wählen, 32 Neuronen in der versteckten Schicht 
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x): # = ich wende NN auf mein Input an 
        x = self.fc1(x) # Matrizenmultiplikation 
        x = self.relu(x) # Aktivierung mit ReLU
        x = self.fc2(x) # Matrizenmultiplikation 
        x = self.sigmoid(x) # Wahrlichkeit-like Aktivierung 
        return x

model = SpamClassifier(input_dim=X_train.shape[1])


# Training 
optimizer = optim.Adam(model.parameters(), lr = 0.01) # Adam -> moderne Version von SGD
lossfn = nn.BCELoss() # Binäre Cross Entropy 

for epoch in range(10): # 10 Epochen Training Loop
    model.train() # Ins Trainingmodus versetzen
    optimizer.zero_grad() # Gradient am Anfang nullen 
    out = model(X_train_tensor) # Batch (gesamter Datensatz wird verwendet)
    loss = lossfn(out, y_train_tensor) # Kostenausrechnen
    loss.backward() # Gradient kalkulieren
    optimizer.step() # Update-Schritt 
    if (epoch+1) % 2 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")
model.eval()
with torch.no_grad():
    preds = model(X_test_tensor)
    predicted = (preds > 0.5).int()
    acc = (predicted.squeeze() == y_test_tensor.squeeze().int()).float().mean().item()
    print(f'Test set accuracy: {acc*100:.2f}%')


def predict_spam(msg):
    x = vectorizer.transform([msg]).toarray()
    x_tensor = torch.tensor(x, dtype=torch.float32)
    model.eval()
    with torch.no_grad():
        pred = model(x_tensor).item()
        return 'SPAM' if pred > 0.5 else 'Kein Spam'

print(predict_spam('Congratulations, you have won a prize!'))
print(predict_spam('Are we meeting at 7pm for dinner?'))

Epoch 2, Loss: 0.6464
Epoch 4, Loss: 0.5205
Epoch 6, Loss: 0.3999
Epoch 8, Loss: 0.2967
Epoch 10, Loss: 0.2153
Test set accuracy: 96.58%
SPAM
Kein Spam
